# SNOOPPI ESM-2 benchmark

Frozen ESM-2 embeddings feed a trainable protein-pair classifier. This notebook reuses the saved train, validation, and test CSVs from the ProtT5 workflow; it does not create new splits. Shared head settings support comparison, but encoder, preprocessing, and input dimensions differ, so this is not an isolated architecture ablation.

All original code cells, including manual and empty cells, remain in their original order with unchanged source. Outputs have been cleared. Run the numbered sections in a Colab runtime with access to `/content/drive/MyDrive/SNOOPPI`, the three split CSVs, and model weights. A GPU is recommended. The sections marked optional can be skipped; no training was rerun during this cleanup.

The main result is the classifier checkpoint selected by validation macro-AP, followed by a validation-selected decision threshold and held-out test metrics.

## 01. Set up the runtime and output paths

Install the listed packages, seed Python/NumPy/PyTorch with 44, and mount Google Drive. Package versions and model revision are unpinned. Paths are rooted at `/content/drive/MyDrive/SNOOPPI`; rerunning writes the same model-tagged artifacts. Defaults: 1,024 residues, embedding batches of 2, cache saves after 1,000 new sequences or at completion, and classifier batches of 1,024.

In [ ]:
%pip install -q -U transformers accelerate safetensors pandas scikit-learn matplotlib

In [ ]:
import os
import random
import re
from contextlib import nullcontext
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from google.colab import drive
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer

SEED = 44

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

try:
    torch.set_float32_matmul_precision("high")
except AttributeError:
    pass

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

In [ ]:
# Mount Google Drive once.
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/SNOOPPI")
SPLITS_DIR = ROOT / "splits"
EMBEDDINGS_DIR = ROOT / "embeddings"
CHECKPOINTS_DIR = ROOT / "checkpoints"
TABLES_DIR = ROOT / "tables"
FIGURES_DIR = ROOT / "figures"

for directory in [
    EMBEDDINGS_DIR,
    CHECKPOINTS_DIR,
    TABLES_DIR,
    FIGURES_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

SPLIT_PATHS = {
    "train": SPLITS_DIR / "train.csv",
    "validation": SPLITS_DIR / "validation.csv",
    "test": SPLITS_DIR / "test.csv",
}

MODEL_ID = "facebook/esm2_t33_650M_UR50D"
MODEL_TAG = "esm2_t33_650M"
MAX_RESIDUES = 1024
EMBEDDING_BATCH_SIZE = 2
CACHE_SAVE_EVERY = 1_000
CLASSIFIER_BATCH_SIZE = 1024

EMBEDDING_CACHE_PATH = (
    EMBEDDINGS_DIR / f"{MODEL_TAG}_snooppi_mean_pool_embeddings.pt"
)
BEST_CHECKPOINT_PATH = (
    CHECKPOINTS_DIR / f"{MODEL_TAG}_best_classifier.pt"
)

print("Training split:", SPLIT_PATHS["train"])
print("Embedding cache:", EMBEDDING_CACHE_PATH)
print("Best checkpoint:", BEST_CHECKPOINT_PATH)

## 02. Load the saved train, validation, and test splits

Read `splits/train.csv`, `splits/validation.csv`, and `splits/test.csv`. An existing numeric `label` column takes precedence; otherwise lowercase `SNOOPPI_final_label` values map `negative` to 0 and `positive` to 1. Unknown/unmapped or other nonbinary labels raise an error, rather than becoming negatives. Missing sequence values also raise an error.

These cells reuse the saved pair-row partitions. They do not verify split hashes, duplicate pairs, protein overlap, or agreement between both label columns; a protein-disjoint evaluation is not established here.

In [ ]:
REQUIRED_COLUMNS = {
    "partner_A_sequence",
    "partner_B_sequence",
}


def load_split(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Keep the same saved split used by ProtT5; "
            "do not silently create a new split for this benchmark."
        )

    dataframe = pd.read_csv(path)
    missing_columns = REQUIRED_COLUMNS - set(dataframe.columns)
    if missing_columns:
        raise ValueError(
            f"{path.name} is missing columns: {sorted(missing_columns)}"
        )

    if "label" not in dataframe.columns:
        if "SNOOPPI_final_label" not in dataframe.columns:
            raise ValueError(
                f"{path.name} needs either 'label' or "
                "'SNOOPPI_final_label'."
            )
        dataframe["label"] = (
            dataframe["SNOOPPI_final_label"]
            .astype(str)
            .str.lower()
            .map({"negative": 0.0, "positive": 1.0})
        )

    if dataframe[list(REQUIRED_COLUMNS)].isna().any().any():
        raise ValueError(f"{path.name} contains missing protein sequences.")

    dataframe["label"] = pd.to_numeric(
        dataframe["label"], errors="raise"
    ).astype(np.float32)

    observed_labels = set(dataframe["label"].unique().tolist())
    if not observed_labels.issubset({0.0, 1.0}):
        raise ValueError(
            f"{path.name} has non-binary labels: {observed_labels}"
        )

    return dataframe.reset_index(drop=True)


train_df = load_split(SPLIT_PATHS["train"])
val_df = load_split(SPLIT_PATHS["validation"])
test_df = load_split(SPLIT_PATHS["test"])

for split_name, dataframe in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df),
]:
    counts = dataframe["label"].value_counts().sort_index().to_dict()
    print(f"{split_name:10s}: {len(dataframe):,} pairs | labels {counts}")

## 03. Load the frozen ESM-2 encoder

Load `facebook/esm2_t33_650M_UR50D`, disable gradients, and use evaluation mode. The encoder produces 1,280 values per protein, giving 3,840 pair features. CUDA inference uses FP16; CPU inference uses FP32. Only the downstream classifier is trained.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model_dtype = torch.float16 if DEVICE.type == "cuda" else torch.float32
encoder = AutoModel.from_pretrained(
    MODEL_ID,
    torch_dtype=model_dtype,
    low_cpu_mem_usage=True,
).to(DEVICE)

encoder.eval()
for parameter in encoder.parameters():
    parameter.requires_grad = False

EMBEDDING_DIM = int(encoder.config.hidden_size)
PAIR_FEATURE_DIM = 3 * EMBEDDING_DIM

print("ESM-2 embedding dimension:", EMBEDDING_DIM)
print("Classifier input dimension:", PAIR_FEATURE_DIM)

## 04. Normalize sequences and build or resume the embedding cache

Uppercase sequences, remove literal spaces, replace noncanonical amino-acid characters with `X`, and retain the first 1,024 residues. Unique raw sequences from all three splits are sorted by normalized length to reduce padding.

Mean-pool the final hidden states over positions allowed by the attention mask and excluded from the tokenizer's special-token mask. Pooling accumulates in FP32; cached CPU vectors are FP16 and keyed by the original sequence. Normalized or truncated collisions are not deduplicated.

The cache is saved through a temporary file and replacement. Reload checks model ID, residue limit, and embedding dimension; it does not check the stored pooling description, normalization version, or model revision. Missing entries resume from the last saved cache; this does not resume classifier training.

In [ ]:
def normalize_sequence(sequence):
    sequence = str(sequence).upper().replace(" ", "")
    # ESM-2 has an unknown-residue token. Map non-canonical residues to X.
    sequence = re.sub(r"[^ACDEFGHIKLMNPQRSTVWY]", "X", sequence)
    return sequence[:MAX_RESIDUES]


all_sequences = pd.unique(
    pd.concat(
        [
            train_df["partner_A_sequence"],
            train_df["partner_B_sequence"],
            val_df["partner_A_sequence"],
            val_df["partner_B_sequence"],
            test_df["partner_A_sequence"],
            test_df["partner_B_sequence"],
        ],
        ignore_index=True,
    )
).tolist()

# Similar lengths in the same mini-batch reduce padding and GPU memory use.
all_sequences.sort(key=lambda sequence: len(normalize_sequence(sequence)))
print(f"Unique protein sequences: {len(all_sequences):,}")

In [ ]:
def atomic_torch_save(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(path.name + ".tmp")
    torch.save(payload, temporary_path)
    os.replace(temporary_path, path)


def load_trusted_torch_file(path, map_location="cpu"):
    # These files are created by this notebook, so full checkpoint loading
    # is intentional. The fallback keeps compatibility with older PyTorch.
    try:
        return torch.load(
            path,
            map_location=map_location,
            weights_only=False,
        )
    except TypeError:
        return torch.load(path, map_location=map_location)


def save_embedding_cache(embedding_cache):
    payload = {
        "model_id": MODEL_ID,
        "max_residues": MAX_RESIDUES,
        "embedding_dim": EMBEDDING_DIM,
        "pooling": "mean over non-special residue tokens",
        "embeddings": embedding_cache,
    }
    atomic_torch_save(payload, EMBEDDING_CACHE_PATH)


def load_embedding_cache():
    if not EMBEDDING_CACHE_PATH.exists():
        return {}

    payload = load_trusted_torch_file(EMBEDDING_CACHE_PATH)
    expected_metadata = {
        "model_id": MODEL_ID,
        "max_residues": MAX_RESIDUES,
        "embedding_dim": EMBEDDING_DIM,
    }

    for key, expected_value in expected_metadata.items():
        if payload.get(key) != expected_value:
            raise ValueError(
                f"Embedding cache mismatch for {key}: "
                f"found {payload.get(key)!r}, expected {expected_value!r}."
            )

    cache = payload["embeddings"]
    print(f"Loaded {len(cache):,} cached ESM-2 embeddings.")
    return cache


@torch.inference_mode()
def embed_missing_sequences(sequences, embedding_cache):
    missing_sequences = [
        sequence for sequence in sequences if sequence not in embedding_cache
    ]

    if not missing_sequences:
        print("Embedding cache is already complete.")
        return embedding_cache

    print(f"Sequences still to embed: {len(missing_sequences):,}")
    new_since_save = 0

    for start in range(0, len(missing_sequences), EMBEDDING_BATCH_SIZE):
        original_batch = missing_sequences[
            start : start + EMBEDDING_BATCH_SIZE
        ]
        cleaned_batch = [
            normalize_sequence(sequence) for sequence in original_batch
        ]

        tokenized = tokenizer(
            cleaned_batch,
            add_special_tokens=True,
            padding=True,
            truncation=True,
            max_length=MAX_RESIDUES + 2,
            return_special_tokens_mask=True,
            return_tensors="pt",
        )

        special_tokens_mask = tokenized.pop("special_tokens_mask").to(DEVICE)
        model_inputs = {
            name: tensor.to(DEVICE)
            for name, tensor in tokenized.items()
        }

        autocast_context = (
            torch.autocast(device_type="cuda", dtype=torch.float16)
            if DEVICE.type == "cuda"
            else nullcontext()
        )

        with autocast_context:
            output = encoder(**model_inputs)

        hidden = output.last_hidden_state.float()
        residue_mask = (
            model_inputs["attention_mask"].bool()
            & ~special_tokens_mask.bool()
        )
        residue_mask_float = residue_mask.unsqueeze(-1).to(hidden.dtype)

        pooled = (
            (hidden * residue_mask_float).sum(dim=1)
            / residue_mask_float.sum(dim=1).clamp(min=1)
        ).cpu().half()

        for sequence, embedding in zip(original_batch, pooled):
            embedding_cache[sequence] = embedding

        new_since_save += len(original_batch)
        completed = start + len(original_batch)

        if (
            new_since_save >= CACHE_SAVE_EVERY
            or completed == len(missing_sequences)
        ):
            save_embedding_cache(embedding_cache)
            new_since_save = 0
            print(
                f"Embedded {completed:,}/{len(missing_sequences):,} "
                f"new sequences; cache saved to Drive."
            )

    return embedding_cache

In [ ]:
embedding_cache = load_embedding_cache()
embedding_cache = embed_missing_sequences(all_sequences, embedding_cache)

missing_after_embedding = [
    sequence for sequence in all_sequences if sequence not in embedding_cache
]
if missing_after_embedding:
    raise RuntimeError(
        f"Embedding cache is incomplete: {len(missing_after_embedding):,} "
        "sequences are missing."
    )

sample_embedding = next(iter(embedding_cache.values()))
if sample_embedding.numel() != EMBEDDING_DIM:
    raise RuntimeError(
        f"Expected {EMBEDDING_DIM} values per embedding, "
        f"found {sample_embedding.numel()}."
    )

print(f"Embedding cache complete: {len(embedding_cache):,} proteins")

### 04a. Optional manual cache save and retained empty cell

The next cell is a historical progress check and manual save of the current in-memory embedding cache. Skip it after the automatic embedding pass has completed. The following empty code cell is retained for source preservation and has no effect.

In [ ]:
print(f"Currently embedded: {len(embedding_cache):,}")
save_embedding_cache(embedding_cache)
print("Progress saved:", EMBEDDING_CACHE_PATH)

## 05. Build symmetric pair features and data loaders

For protein vectors `a` and `b`, concatenate `[a + b, abs(a - b), a * b]`. This representation is unchanged when partners are swapped. FP16 cached vectors are converted to FP32. Training batches are shuffled with a seeded generator; validation and test retain CSV row order.

In [ ]:
class PPIPairDataset(Dataset):
    def __init__(self, dataframe, embedding_cache):
        self.dataframe = dataframe.reset_index(drop=True)
        self.embedding_cache = embedding_cache

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        embedding_a = self.embedding_cache[
            row["partner_A_sequence"]
        ].float()
        embedding_b = self.embedding_cache[
            row["partner_B_sequence"]
        ].float()

        pair_features = torch.cat(
            [
                embedding_a + embedding_b,
                torch.abs(embedding_a - embedding_b),
                embedding_a * embedding_b,
            ]
        )
        label = torch.tensor(row["label"], dtype=torch.float32)
        return pair_features, label


train_dataset = PPIPairDataset(train_df, embedding_cache)
val_dataset = PPIPairDataset(val_df, embedding_cache)
test_dataset = PPIPairDataset(test_df, embedding_cache)

loader_settings = {
    "batch_size": CLASSIFIER_BATCH_SIZE,
    "num_workers": 0,
    "pin_memory": DEVICE.type == "cuda",
}

train_loader = DataLoader(
    train_dataset,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
    **loader_settings,
)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_settings)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_settings)

features, labels = next(iter(train_loader))
print("Feature batch:", tuple(features.shape))
print("Label batch:", tuple(labels.shape))
assert features.shape[1] == PAIR_FEATURE_DIM

## 06. Define the classifier and weighted objective

The head is `3840 → 512 → 512 → 256 → 1`, with ReLU and dropout 0.1 after each hidden layer. AdamW uses learning rate `1e-4` and weight decay `1e-4`. Weighted binary cross-entropy uses training negatives divided by training positives as `pos_weight`. Sigmoid outputs are interaction scores; calibration or binding affinity is not established.

In [ ]:
class PPIClassifier(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()

        self.embedding_dim = int(embedding_dim)
        input_dimension = 3 * self.embedding_dim

        self.network = nn.Sequential(
            nn.Linear(input_dimension, 512),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(256, 1),
        )

    def forward(self, features):
        return self.network(features).squeeze(-1)


model = PPIClassifier(EMBEDDING_DIM).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4,
)

n_positive = int(train_df["label"].sum())
n_negative = len(train_df) - n_positive
positive_weight = n_negative / n_positive

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([positive_weight], device=DEVICE)
)

print(model)
print(f"Positive weight: {positive_weight:.6f}")

## 07. Train and save the best validation checkpoint

Train for at most 50 epochs, stopping after 10 consecutive epochs without a strict improvement in validation macro-AP. Macro-AP is the mean of positive-class and negative-class `average_precision_score`; it is not positive-only AP. Epoch reports also include weighted loss and AUROC. Training epoch metrics use batches scored while parameters are changing.

History is saved every epoch. Each improvement immediately replaces the best checkpoint, including classifier and optimizer states, selected epoch, validation metrics, and selected configuration fields. The loop does not restore those states, RNG state, or a last-epoch checkpoint; rerunning it resets history and selection counters and uses the current in-memory model.

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    all_labels = []
    all_probabilities = []

    gradient_context = torch.enable_grad() if training else torch.no_grad()
    with gradient_context:
        for features, labels in loader:
            features = features.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            if training:
                optimizer.zero_grad(set_to_none=True)

            logits = model(features)
            loss = criterion(logits, labels)

            if training:
                loss.backward()
                optimizer.step()

            probabilities = torch.sigmoid(logits)
            total_loss += loss.item() * labels.size(0)
            all_labels.append(labels.detach().cpu().numpy())
            all_probabilities.append(
                probabilities.detach().cpu().numpy()
            )

    labels = np.concatenate(all_labels).astype(int)
    probabilities = np.concatenate(all_probabilities)
    positive_ap = average_precision_score(labels, probabilities)
    negative_ap = average_precision_score(1 - labels, 1 - probabilities)

    return {
        "loss": total_loss / len(loader.dataset),
        "auroc": roc_auc_score(labels, probabilities),
        "positive_ap": positive_ap,
        "negative_ap": negative_ap,
        "macro_ap": (positive_ap + negative_ap) / 2,
    }

In [ ]:
MAX_EPOCHS = 50
PATIENCE = 10

history = []
best_val_macro_ap = -np.inf
epochs_without_improvement = 0

for epoch in range(1, MAX_EPOCHS + 1):
    train_metrics = run_epoch(
        model,
        train_loader,
        criterion,
        optimizer=optimizer,
    )
    val_metrics = run_epoch(model, val_loader, criterion)

    epoch_record = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_auroc": train_metrics["auroc"],
        "train_macro_ap": train_metrics["macro_ap"],
        "val_loss": val_metrics["loss"],
        "val_auroc": val_metrics["auroc"],
        "val_macro_ap": val_metrics["macro_ap"],
    }
    history.append(epoch_record)

    print(
        f"Epoch {epoch:02d} | "
        f"train loss {train_metrics['loss']:.4f} | "
        f"val loss {val_metrics['loss']:.4f} | "
        f"val AUROC {val_metrics['auroc']:.4f} | "
        f"val macro-AP {val_metrics['macro_ap']:.4f}"
    )

    # Persist history after every epoch in case the Colab runtime stops.
    pd.DataFrame(history).to_csv(
        TABLES_DIR / f"{MODEL_TAG}_training_history.csv",
        index=False,
    )

    if val_metrics["macro_ap"] > best_val_macro_ap:
        best_val_macro_ap = val_metrics["macro_ap"]
        epochs_without_improvement = 0

        checkpoint = {
            "epoch": epoch,
            "model_id": MODEL_ID,
            "embedding_dim": EMBEDDING_DIM,
            "pair_feature_dim": PAIR_FEATURE_DIM,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_val_macro_ap": float(best_val_macro_ap),
            "validation_metrics": val_metrics,
            "selection_metric": "validation macro-AP",
            "seed": SEED,
            "max_residues": MAX_RESIDUES,
            "pair_features": ["sum", "absolute_difference", "product"],
            "classifier_architecture": str(model),
        }
        atomic_torch_save(checkpoint, BEST_CHECKPOINT_PATH)
        print(f"  Saved new best checkpoint: {BEST_CHECKPOINT_PATH}")
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping after {epoch} epochs.")
        break

history_df = pd.DataFrame(history)
print(
    f"Best validation macro-AP: {best_val_macro_ap:.4f} | "
    f"checkpoint exists: {BEST_CHECKPOINT_PATH.exists()}"
)

## 08. Reload the selected checkpoint and evaluate

Reload the best classifier, then select the threshold that maximizes validation macro-F1 over 181 values from 0.05 to 0.95; the first threshold wins ties. Apply that threshold to all three splits. Test data do not select either checkpoint or threshold.

Export weighted loss, AUROC, per-class AP, macro-AP, macro-F1, MCC, balanced accuracy, and positive-class precision/recall. Test prediction rows follow the test CSV and include its zero-based row index. Save the test confusion matrix, append threshold and final metrics to the best checkpoint, and write classifier weights separately.

In [ ]:
best_checkpoint = load_trusted_torch_file(
    BEST_CHECKPOINT_PATH,
    map_location=DEVICE,
)
model.load_state_dict(best_checkpoint["model_state_dict"])
model.eval()


def collect_predictions(model, loader, criterion):
    labels_all = []
    probabilities_all = []
    total_loss = 0.0

    with torch.no_grad():
        for features, labels in loader:
            features = features.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            logits = model(features)
            loss = criterion(logits, labels)
            probabilities = torch.sigmoid(logits)

            total_loss += loss.item() * labels.size(0)
            labels_all.append(labels.cpu().numpy())
            probabilities_all.append(probabilities.cpu().numpy())

    return (
        total_loss / len(loader.dataset),
        np.concatenate(labels_all).astype(int),
        np.concatenate(probabilities_all),
    )


def summarize_predictions(loss, labels, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)
    positive_ap = average_precision_score(labels, probabilities)
    negative_ap = average_precision_score(1 - labels, 1 - probabilities)

    return {
        "loss": loss,
        "auroc": roc_auc_score(labels, probabilities),
        "positive_ap": positive_ap,
        "negative_ap": negative_ap,
        "macro_ap": (positive_ap + negative_ap) / 2,
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "mcc": matthews_corrcoef(labels, predictions),
        "balanced_accuracy": balanced_accuracy_score(labels, predictions),
        "precision": precision_score(
            labels, predictions, zero_division=0
        ),
        "recall": recall_score(labels, predictions, zero_division=0),
    }


train_loss, train_labels, train_probabilities = collect_predictions(
    model, train_loader, criterion
)
val_loss, val_labels, val_probabilities = collect_predictions(
    model, val_loader, criterion
)
test_loss, test_labels, test_probabilities = collect_predictions(
    model, test_loader, criterion
)

# Select the classification threshold using validation data only.
threshold_grid = np.linspace(0.05, 0.95, 181)
decision_threshold = max(
    threshold_grid,
    key=lambda threshold: f1_score(
        val_labels,
        val_probabilities >= threshold,
        average="macro",
    ),
)

split_results = {
    "Train": summarize_predictions(
        train_loss,
        train_labels,
        train_probabilities,
        decision_threshold,
    ),
    "Validation": summarize_predictions(
        val_loss,
        val_labels,
        val_probabilities,
        decision_threshold,
    ),
    "Test": summarize_predictions(
        test_loss,
        test_labels,
        test_probabilities,
        decision_threshold,
    ),
}

split_metrics = (
    pd.DataFrame(split_results)
    .T
    .rename_axis("split")
    .reset_index()
)
split_metrics.to_csv(
    TABLES_DIR / f"{MODEL_TAG}_split_metrics.csv",
    index=False,
)

test_predictions = pd.DataFrame(
    {
        "row_index": np.arange(len(test_labels)),
        "label": test_labels,
        "interaction_score": test_probabilities,
        "prediction": (
            test_probabilities >= decision_threshold
        ).astype(int),
    }
)
test_predictions.to_csv(
    TABLES_DIR / f"{MODEL_TAG}_test_predictions.csv",
    index=False,
)

test_confusion_matrix = confusion_matrix(
    test_labels,
    test_predictions["prediction"],
    labels=[0, 1],
)
pd.DataFrame(
    test_confusion_matrix,
    index=["true_negative", "true_positive"],
    columns=["predicted_negative", "predicted_positive"],
).to_csv(TABLES_DIR / f"{MODEL_TAG}_test_confusion_matrix.csv")

# Add the validation-selected threshold and final metrics to the saved
# best checkpoint, then write a separate weights-only file.
best_checkpoint["decision_threshold"] = float(decision_threshold)
best_checkpoint["split_metrics"] = split_metrics.set_index(
    "split"
).to_dict("index")
atomic_torch_save(best_checkpoint, BEST_CHECKPOINT_PATH)
atomic_torch_save(
    model.state_dict(),
    CHECKPOINTS_DIR / f"{MODEL_TAG}_best_classifier_weights.pt",
)

display(split_metrics.round(4))
print(f"Validation-selected threshold: {decision_threshold:.3f}")
print(f"Selected epoch: {best_checkpoint['epoch']}")

## 09. Export training and evaluation figures

Save training/validation curves and the test confusion matrix as PNGs under `figures/`. These cells require `history_df` from the training loop and evaluation results from Section 08; loading a checkpoint alone does not recreate the history.

In [ ]:
# Section 9: Save training and evaluation figures

import matplotlib.pyplot as plt

plt.rcParams.update(
    {
        "font.family": "serif",
        "font.size": 11,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.dpi": 150,
    }
)

selected_epoch = int(best_checkpoint["epoch"])


# ============================================================
# Figure 1: Training curves and validation performance
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.3))

# Training and validation loss
axes[0].plot(
    history_df["epoch"],
    history_df["train_loss"],
    linewidth=2,
    label="Training",
)

axes[0].plot(
    history_df["epoch"],
    history_df["val_loss"],
    linewidth=2,
    label="Validation",
)

axes[0].axvline(
    selected_epoch,
    color="black",
    linestyle="--",
    linewidth=1,
    label=f"Selected epoch: {selected_epoch}",
)

axes[0].set(
    title="ESM-2 training and validation loss",
    xlabel="Epoch",
    ylabel="Weighted binary cross-entropy",
)

axes[0].grid(alpha=0.2)
axes[0].legend(frameon=False)


# Validation AUROC and macro-AP
axes[1].plot(
    history_df["epoch"],
    history_df["val_auroc"],
    linewidth=2,
    label="AUROC",
)

axes[1].plot(
    history_df["epoch"],
    history_df["val_macro_ap"],
    linewidth=2,
    label="Macro-AP",
)

axes[1].axvline(
    selected_epoch,
    color="black",
    linestyle="--",
    linewidth=1,
)

axes[1].set(
    title="ESM-2 validation performance",
    xlabel="Epoch",
    ylabel="Score",
    ylim=(0.5, 1.0),
)

axes[1].grid(alpha=0.2)
axes[1].legend(frameon=False)

fig.tight_layout()

training_curve_path = (
    FIGURES_DIR / f"{MODEL_TAG}_training_curves.png"
)

fig.savefig(
    training_curve_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

print(f"Saved training curves to: {training_curve_path}")


# ============================================================
# Figure 2: Test confusion matrix
# ============================================================

fig, ax = plt.subplots(figsize=(5.2, 4.5))

image = ax.imshow(
    test_confusion_matrix,
    cmap="Blues",
)

maximum_count = test_confusion_matrix.max()

for row in range(2):
    for column in range(2):
        value = test_confusion_matrix[row, column]

        ax.text(
            column,
            row,
            f"{value:,}",
            ha="center",
            va="center",
            fontsize=12,
            color=(
                "white"
                if value > maximum_count / 2
                else "black"
            ),
        )

ax.set(
    xticks=[0, 1],
    yticks=[0, 1],
    xticklabels=[
        "Non-interaction",
        "Interaction",
    ],
    yticklabels=[
        "Non-interaction",
        "Interaction",
    ],
    xlabel="Predicted class",
    ylabel="True class",
    title=(
        "ESM-2 test confusion matrix\n"
        f"Threshold = {decision_threshold:.2f}"
    ),
)

fig.colorbar(
    image,
    ax=ax,
    fraction=0.046,
    pad=0.04,
)

fig.tight_layout()

confusion_matrix_path = (
    FIGURES_DIR / f"{MODEL_TAG}_test_confusion_matrix.png"
)

fig.savefig(
    confusion_matrix_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

print(f"Saved confusion matrix to: {confusion_matrix_path}")

## 10. Check saved artifacts

Print existence checks for the embedding cache, two classifier files, four CSV tables, and two figures. Only a missing best-checkpoint file raises an error; these checks do not verify artifact contents or cross-run consistency.

In [ ]:
expected_outputs = [
    EMBEDDING_CACHE_PATH,
    BEST_CHECKPOINT_PATH,
    CHECKPOINTS_DIR / f"{MODEL_TAG}_best_classifier_weights.pt",
    TABLES_DIR / f"{MODEL_TAG}_training_history.csv",
    TABLES_DIR / f"{MODEL_TAG}_split_metrics.csv",
    TABLES_DIR / f"{MODEL_TAG}_test_predictions.csv",
    TABLES_DIR / f"{MODEL_TAG}_test_confusion_matrix.csv",
    FIGURES_DIR / f"{MODEL_TAG}_training_curves.png",
    FIGURES_DIR / f"{MODEL_TAG}_test_confusion_matrix.png",
]

for output_path in expected_outputs:
    print(f"{output_path.exists()!s:5s} | {output_path}")

if not BEST_CHECKPOINT_PATH.exists():
    raise RuntimeError("The best checkpoint was not saved successfully.")

## 11. Optional Colab figure downloads

The final cell downloads the two figure PNGs through the Colab browser interface. Skip it when browser downloads are unnecessary. Its paths are hardcoded to the original Drive location.

In [ ]:
from google.colab import files

files.download(
    "/content/drive/MyDrive/SNOOPPI/figures/"
    "esm2_t33_650M_training_curves.png"
)

files.download(
    "/content/drive/MyDrive/SNOOPPI/figures/"
    "esm2_t33_650M_test_confusion_matrix.png"
)